# Model Experiments

This notebook allows you to experiment with different models and compare their performance.

## Steps:
1. Load and preprocess data
2. Try different models
3. Compare performance
4. Select best model

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

# Add src to path
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from data_processing import DataProcessor
from model_training import ModelTrainer

In [ ]:
# Load data
df = pd.read_csv('../data/raw/user_data.csv')
target_col = 'investment_success'  # Change to your target column

print(f"Data shape: {df.shape}")
print(f"Target column: {target_col}")

In [ ]:
# Preprocess data
processor = DataProcessor()
X_processed, y_processed = processor.preprocess(
    df, 
    target_column=target_col,
    missing_strategy='mean',
    scaler_type='standard',
    fit=True
)

print(f"Processed features shape: {X_processed.shape}")
print(f"Target shape: {y_processed.shape}")

In [ ]:
# Experiment with different models
models_to_try = ['random_forest', 'logistic']  # Add 'linear' for regression
results = {}

for model_name in models_to_try:
    print(f"\nTraining {model_name}...")
    trainer = ModelTrainer()
    trainer.create_model(model_name, task_type='classification')
    metrics = trainer.train(X_processed, y_processed, test_size=0.2)
    results[model_name] = metrics
    print(f"Test Accuracy: {metrics.get('test_accuracy', 'N/A'):.4f}")

In [ ]:
# Compare results
comparison_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(comparison_df[['test_accuracy', 'test_f1', 'cv_mean']])

In [ ]:
# Feature importance for best model
best_model_name = comparison_df['test_accuracy'].idxmax()
print(f"\nBest model: {best_model_name}")

# Retrain best model
trainer = ModelTrainer()
trainer.create_model(best_model_name, task_type='classification')
trainer.train(X_processed, y_processed)

# Get feature importance
importance = trainer.get_feature_importance()
importance_df = pd.DataFrame(list(importance.items()), columns=['Feature', 'Importance'])
importance_df = importance_df.sort_values('Importance', ascending=False)
print("\nTop Features:")
print(importance_df.head(10))